<a href="https://colab.research.google.com/github/iamtrask/abcGPT/blob/main/notebooks/train_dual_xl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# abcGPT dual-source — XL upgrade (bigger model + longer training)

alt_mixed Uniform with the cheap improvements bundled:
- `n_layer=8, n_embd=512` (~50M total params, 2.4x previous; each slot is ~25M, vs ~10M before)
- `max_iters=24000` (2x previous)
- `dropout=0.05` (the dual scheme already regularizes via alpha-stochasticity)
- `learning_rate=8e-4` (conventional for bigger models)
- `gradient_accumulation_steps=2` (effective batch 128)

Goal: close the gap to per-corpus single-model baselines. Previous 21M alt_mixed lands at wiki val ~1.78. This should hit ~1.50-1.60.

**Just the training**, no analysis prelude. Hit Run All. Compute estimate: ~5-6 hours on T4.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install --quiet zstandard tiktoken

In [ ]:
import os
if not os.path.exists('/content/abcGPT'):
    !git clone --depth 1 https://github.com/iamtrask/abcGPT.git /content/abcGPT
else:
    !cd /content/abcGPT && git pull --rebase --autostash
%cd /content/abcGPT

In [ ]:
!python data/shakespeare_wiki_char/prepare.py

## Run directory (tagged `-xl` so this run is distinct from the others)

In [ ]:
RUN_ID = None   # set explicitly to resume, e.g. "20260519-XXXXXX-xl"

In [ ]:
import time, os, glob
DRIVE_ROOT = '/content/drive/MyDrive/abcGPT/runs'
os.makedirs(DRIVE_ROOT, exist_ok=True)
if 'RUN_ID' not in dir() or RUN_ID is None:
    RUN_ID = time.strftime('%Y%m%d-%H%M%S') + '-xl'
elif not RUN_ID.endswith('-xl'):
    RUN_ID = RUN_ID + '-xl'
OUT_DIR_DUAL = f'{DRIVE_ROOT}/{RUN_ID}'
os.makedirs(OUT_DIR_DUAL, exist_ok=True)
print('run dir:', OUT_DIR_DUAL)
snaps = sorted(glob.glob(os.path.join(OUT_DIR_DUAL, '*.pt.zst')))
print(f'  {len(snaps)} existing snapshots' + (' (will resume)' if snaps else ' (fresh run)'))

## Train

On a T4 GPU expect roughly 5-6 hours for 24000 iters with the bigger model. Pass time will be ~75s/pass (vs ~40s for the 21M model).

In [ ]:
!python train_dual.py config/train_shakespeare_wiki_dual_xl.py \
    --out_dir=$OUT_DIR_DUAL \
    --batch_mode=alt_mixed \
    --first_pass_corpus=shake \
    --mix_distribution=uniform